In [17]:


### --- Importacoes --- ###

import pandas as pd
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import seaborn as sns
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import load_model
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix

## ETAPA 1: Parâmetros de Experimentação e Funções Auxiliares

Definição de todos os parâmetros que queremos testar no loop agrupando todas as funções de criação de modelo e pré-processamento.

In [ ]:
### --- Parâmetros de Experimentação --- ###

PASTA_BASE_EXPERIMENTOS = "meus_experimentos_cnn"

# Adicione os modelos que você quer testar

#  'resnet50', 'EfficientNetB0', 'VGG16'
PARAM_MODELOS = ['efficientnet'] 


# Listas de parâmetros
PARAM_BATCH_SIZES = [32]
PARAM_EPOCHS = [40, 60]
PARAM_NUM_AMOSTRAS_PER_CLASSE = [1000]


### --- Constantes do Projeto --- ###

#max_len 175
MAX_LEN = 175 
SAMPLE_RATE = 22050
N_MELS = 128
selected_class = ['guitar', 'organ', 'flute', 'string', 'bass', 'reed', 'vocal', 'synth_lead', 'brass']
NUM_CLASSES = len(selected_class)

In [19]:
### --- Função de Pré-processamento --- ###

def extrair_features(file_path):
    """
    Função para carregar um arquivo de áudio, extrair o espectrograma em Mel 
    e padronizar seu tamanho para MAX_LEN.
    """
    try:
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
        spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
        log_spectrogram = librosa.power_to_db(spectrogram, ref=np.max)
        
        if log_spectrogram.shape[1] < MAX_LEN:
            pad_width = MAX_LEN - log_spectrogram.shape[1]
            padded_spec = np.pad(log_spectrogram, pad_width=((0, 0), (0, pad_width)), mode='constant')
            return padded_spec
        else:
            return log_spectrogram[:, :MAX_LEN]
            
    except Exception as e:
        print(f"Erro ao processar o arquivo {file_path}: {e}")
        return None

### --- Funções de Criação de Modelo --- ###

def criar_modelo_baseline(input_shape, num_classes):
    model = Sequential([
        Input(shape=input_shape),
        Conv2D(32, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(128, kernel_size=(3, 3), activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'], run_eagerly=False)
    return model

def criar_modelo_resnet50(input_shape_rgb, num_classes):
    base_model_resnet = ResNet50(include_top=False, weights='imagenet', input_shape=input_shape_rgb)
    base_model_resnet.trainable = False
    model = Sequential([
        Input(shape=input_shape_rgb),
        base_model_resnet,
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'], run_eagerly=False)
    return model

def criar_modelo_efficientnet(input_shape_rgb, num_classes):
    base_model_effnet = EfficientNetB0(include_top=False, weights='imagenet', input_shape=input_shape_rgb)
    base_model_effnet.trainable = False
    model = Sequential([
        Input(shape=input_shape_rgb),
        base_model_effnet,
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'], run_eagerly=False)
    return model

def criar_modelo_vgg16(input_shape_rgb, num_classes):
    base_model_vgg = VGG16(include_top=False, weights='imagenet', input_shape=input_shape_rgb)
    base_model_vgg.trainable = False
    model = Sequential([
        Input(shape=input_shape_rgb),
        base_model_vgg,
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'], run_eagerly=False)
    return model

### --- Callback Customizado --- ###
class EagerTensorFixCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None:
            for key, value in logs.items():
                if isinstance(value, tf.Tensor):
                    logs[key] = value.numpy().item()

## ETAPA 2: Carregamento do DataFrame Principal

Carrega o JSON uma vez e filtramos as classes de interesse. Também prepare-se os 'Encoders' que serão usados em cada loop.

In [20]:
audios_path = 'data\\nsynth-train\\audio'
json_archive = 'data\\nsynth-train\\examples.json'

df = pd.read_json(json_archive)
df = df.T
df['path'] = df['note_str'].apply(lambda name: os.path.join(audios_path, f'{name}.wav'))

# Cria o dataframe filtrado com todas as amostras das classes selecionadas
df_filtred = df[df['instrument_family_str'].isin(selected_class)]

print("DataFrame principal carregado e filtrado.")
print(f"Total de amostras nas classes selecionadas: {len(df_filtred)}")
print("\nDistribuição total das classes selecionadas:")
print(df_filtred['instrument_family_str'].value_counts())

DataFrame principal carregado e filtrado.
Total de amostras nas classes selecionadas: 203183

Distribuição total das classes selecionadas:
instrument_family_str
bass          65474
organ         34477
guitar        32690
string        19474
reed          13911
brass         12675
vocal         10208
flute          8773
synth_lead     5501
Name: count, dtype: int64


In [21]:
# --- Preparar os Encoders --- 
label_encoder = LabelEncoder()
onehot_encoder = OneHotEncoder(sparse_output=False)

print("Fitando LabelEncoder e OneHotEncoder...")
# Pega todos os labels possíveis do dataframe filtrado
all_possible_labels = df_filtred['instrument_family_str'].values

# Fit LabelEncoder
integer_encoded_full = label_encoder.fit_transform(all_possible_labels)

# Fit OneHotEncoder
integer_encoded_full = integer_encoded_full.reshape(len(integer_encoded_full), 1)
onehot_encoder.fit(integer_encoded_full)

class_names = label_encoder.classes_
print(f"\nClasses codificadas: {class_names}")

import joblib

caminho_le = "label_encoder.joblib"
joblib.dump(label_encoder, caminho_le)

print(f"Label Encoder salvo em: {caminho_le}")

Fitando LabelEncoder e OneHotEncoder...

Classes codificadas: ['bass' 'brass' 'flute' 'guitar' 'organ' 'reed' 'string' 'synth_lead'
 'vocal']
Label Encoder salvo em: label_encoder.joblib


## ETAPA 3: Loop de Treinamento e Experimentação

Iteraração por todas as combinações de parâmetros definidas na ETAPA 1.

Para cada combinação:
1.  Limpar a memória da GPU (`clear_session()`).
2.  Criar uma pasta única para o experimento (ex: `meus_experimentos_cnn/run_baseline...`).
3.  Criar o `mini_df` com o `n_samples` da vez.
4.  Extrair as features *apenas* para esse `mini_df`.
5.  Dividir em treino/validação.
6.  Criar e compilar o modelo da vez.
7.  Treinar o modelo, salvando o **melhor** modelo (`modelo_melhor.h5`) nessa pasta.
8.  Salvar o modelo **final** (`modelo_final.h5`) nessa pasta.
9.  Salvar os gráficos de acurácia e perda (`metricas_treinamento.png`) nessa pasta.

In [22]:
print("Iniciando loop de experimentos...")
tqdm.pandas()

for n_samples in PARAM_NUM_AMOSTRAS_PER_CLASSE:
    for batch in PARAM_BATCH_SIZES:
        for epochs in PARAM_EPOCHS:
            for modelo_tipo in PARAM_MODELOS:
                
                # --- 1: Limpar a sessão e criar pasta ---
                tf.keras.backend.clear_session() 
                run_name = f"run_modelo_{modelo_tipo}_amostras_{n_samples}_batch_{batch}_epochs_{epochs}"
                output_dir = os.path.join(PASTA_BASE_EXPERIMENTOS, run_name)
                os.makedirs(output_dir, exist_ok=True)
                
                print(f"\n{'-'*70}")
                print(f"EXECUTANDO: {run_name}")
                print(f"Salvando resultados em: {output_dir}")
                print(f"{'='*70}")

                # --- 2: Preparar os dados para ESTE run ---
                try:
                    print(f"Preparando {n_samples} amostras por classe...")
                    mini_df = df_filtred.groupby('instrument_family_str').sample(n=n_samples, random_state=42).reset_index(drop=True)
                except ValueError as e:
                    print(f"ERRO: Não foi possível amostrar {n_samples}. Talvez uma classe tenha menos amostras que isso. Pulando... ({e})")
                    continue 
                
                print(f"Extraindo features para {len(mini_df)} arquivos...")
                mini_df['features'] = mini_df['path'].progress_apply(extrair_features)
                mini_df.dropna(inplace=True)
                
                X = np.array(mini_df['features'].tolist())
                X = X[..., np.newaxis]
                labels = mini_df['instrument_family_str'].values
                
                integer_encoded = label_encoder.transform(labels) 
                integer_encoded = integer_encoded.reshape(len(integer_encoded), 1)
                y = onehot_encoder.transform(integer_encoded) 
                
                X_train, X_val, y_train, y_val = train_test_split(
                    X, y, test_size=0.2, stratify=y, random_state=42
                )

                # --- 3: Criar e compilar o modelo ---
                model = None
                if modelo_tipo == 'baseline':
                    print("Criando modelo Baseline (CNN Simples)...")
                    input_shape = X_train.shape[1:]
                    model = criar_modelo_baseline(input_shape, NUM_CLASSES)
                    X_train_run, X_val_run = X_train, X_val
                
                elif modelo_tipo == 'resnet50':
                    print("Criando modelo Intermediário (ResNet50)...")
                    X_train_rgb = np.repeat(X_train, 3, -1)
                    X_val_rgb = np.repeat(X_val, 3, -1)
                    input_shape_rgb = X_train_rgb.shape[1:]
                    model = criar_modelo_resnet50(input_shape_rgb, NUM_CLASSES)
                    X_train_run, X_val_run = X_train_rgb, X_val_rgb
                
                elif modelo_tipo == 'efficientnet':
                    print("Criando modelo SOTA (EfficientNetB0)...")
                    X_train_rgb = np.repeat(X_train, 3, -1)
                    X_val_rgb = np.repeat(X_val, 3, -1)
                    input_shape_rgb = X_train_rgb.shape[1:]
                    model = criar_modelo_efficientnet(input_shape_rgb, NUM_CLASSES)
                    X_train_run, X_val_run = X_train_rgb, X_val_rgb
                
                elif modelo_tipo == 'vgg16':
                    print("Criando modelo SOTA Alternativo (VGG16)...")
                    X_train_rgb = np.repeat(X_train, 3, -1)
                    X_val_rgb = np.repeat(X_val, 3, -1)
                    input_shape_rgb = X_train_rgb.shape[1:]
                    model = criar_modelo_vgg16(input_shape_rgb, NUM_CLASSES)
                    X_train_run, X_val_run = X_train_rgb, X_val_rgb
                

                if model is None:
                    print(f"ERRO: Modelo tipo '{modelo_tipo}' não reconhecido. Pulando...")
                    continue

                # --- 4: Callbacks para ESTE run ---
                best_model_path = os.path.join(output_dir, "modelo_final_weights.h5")
                callbacks_run = [
                    EagerTensorFixCallback(),
                    ModelCheckpoint(filepath=best_model_path, 
                                    save_best_only=True, 
                                    monitor='val_accuracy', 
                                    mode='max', 
                                    verbose=0,
                                    save_weights_only=True),
                    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
                ]

                # --- 5: Treinar o modelo ---
                print(f"Treinando com batch_size={batch}, epochs={epochs}...")
                history = model.fit(
                    X_train_run, y_train,
                    batch_size=batch,
                    epochs=epochs,
                    validation_data=(X_val_run, y_val),
                    callbacks=callbacks_run,
                    verbose=1
                )
                print("Treinamento concluído.")

                # --- 6: Salvar o modelo FINAL ---
                final_model_path = os.path.join(output_dir, "modelo_melhor.h5")
                model.save_weights(final_model_path)
                #model.save(final_model_path)
                print(f"Modelo final salvo em: {final_model_path}")
                print(f"Melhor modelo (checkpoint) salvo em: {best_model_path}")

                # --- 7: Plotar e salvar as métricas ---
                print("Gerando gráficos de métricas...")
                acc = history.history['accuracy']
                val_acc = history.history['val_accuracy']
                loss = history.history['loss']
                val_loss = history.history['val_loss']
                
                plt.figure(figsize=(14, 6))
                
                # Gráfico 1: Acurácia
                plt.subplot(1, 2, 1)
                plt.plot(acc, label='Acurácia (Treino)')
                plt.plot(val_acc, label='Acurácia (Validação)')
                plt.title(f'Acurácia (Batch: {batch}, Amostras: {n_samples})')
                plt.xlabel('Época')
                plt.ylabel('Acurácia')
                plt.legend()
                
                # Gráfico 2: Perda (Loss)
                plt.subplot(1, 2, 2)
                plt.plot(loss, label='Perda (Treino)')
                plt.plot(val_loss, label='Perda (Validação)')
                plt.title(f'Perda (Batch: {batch}, Amostras: {n_samples})')
                plt.xlabel('Época')
                plt.ylabel('Perda')
                plt.legend()
                
                plt.suptitle(f"Resultados: {run_name}", fontsize=16)
                plt.tight_layout(rect=[0, 0.03, 1, 0.95])
                
                plot_save_path = os.path.join(output_dir, "metricas_treinamento.png")
                plt.savefig(plot_save_path)
                plt.close()
                
                print(f"Gráficos salvos em: {plot_save_path}")

                # --- 8: LIMPEZA DE MEMÓRIA ---
                print(f"Limpando memória antes do próximo run...")
                del model
                del history
                del X
                del y
                del X_train
                del X_val
                del y_train
                del y_val
                del X_train_run
                del X_val_run
                
                # Limpa as variáveis RGB se elas existirem
                if 'X_train_rgb' in locals():
                    del X_train_rgb
                if 'X_val_rgb' in locals():
                    del X_val_rgb
                    
                import gc
                gc.collect()
                
                # Força a limpeza da sessão Keras/TF
                tf.keras.backend.clear_session()
                
                print(f"Run {run_name} finalizado e memória limpa.")
                # --- FIM DA LIMPEZA ---
                
print("\n--- Todos os experimentos foram concluídos! ---")

Iniciando loop de experimentos...

----------------------------------------------------------------------
EXECUTANDO: run_modelo_efficientnet_amostras_1000_batch_32_epochs_40
Salvando resultados em: meus_experimentos_cnn\run_modelo_efficientnet_amostras_1000_batch_32_epochs_40
Preparando 1000 amostras por classe...
Extraindo features para 9000 arquivos...


100%|██████████| 9000/9000 [02:37<00:00, 57.22it/s]


Criando modelo SOTA (EfficientNetB0)...
Treinando com batch_size=32, epochs=40...
Epoch 1/40
225/225 [==============================] - 15s 46ms/step - loss: 2.1076 - accuracy: 0.2039 - val_loss: 2.0096 - val_accuracy: 0.2361
Epoch 2/40
225/225 [==============================] - 9s 41ms/step - loss: 2.0324 - accuracy: 0.2267 - val_loss: 1.9347 - val_accuracy: 0.2656
Epoch 3/40
225/225 [==============================] - 9s 41ms/step - loss: 1.9872 - accuracy: 0.2424 - val_loss: 1.9303 - val_accuracy: 0.2767
Epoch 4/40
225/225 [==============================] - 9s 41ms/step - loss: 1.9596 - accuracy: 0.2586 - val_loss: 1.8700 - val_accuracy: 0.3333
Epoch 5/40
225/225 [==============================] - 9s 41ms/step - loss: 1.9415 - accuracy: 0.2631 - val_loss: 1.8556 - val_accuracy: 0.2717
Epoch 6/40
225/225 [==============================] - 10s 43ms/step - loss: 1.9280 - accuracy: 0.2808 - val_loss: 1.8256 - val_accuracy: 0.3128
Epoch 7/40
225/225 [==============================] - 10s 

100%|██████████| 9000/9000 [02:32<00:00, 58.89it/s]


Criando modelo SOTA (EfficientNetB0)...
Treinando com batch_size=32, epochs=60...
Epoch 1/60
225/225 [==============================] - 15s 48ms/step - loss: 2.1162 - accuracy: 0.1907 - val_loss: 2.0023 - val_accuracy: 0.2200
Epoch 2/60
225/225 [==============================] - 9s 41ms/step - loss: 2.0295 - accuracy: 0.2288 - val_loss: 1.9498 - val_accuracy: 0.2839
Epoch 3/60
225/225 [==============================] - 9s 42ms/step - loss: 1.9924 - accuracy: 0.2414 - val_loss: 1.9295 - val_accuracy: 0.2906
Epoch 4/60
225/225 [==============================] - 9s 42ms/step - loss: 1.9575 - accuracy: 0.2603 - val_loss: 1.8783 - val_accuracy: 0.2678
Epoch 5/60
225/225 [==============================] - 10s 45ms/step - loss: 1.9428 - accuracy: 0.2744 - val_loss: 1.8690 - val_accuracy: 0.3083
Epoch 6/60
225/225 [==============================] - 11s 47ms/step - loss: 1.9293 - accuracy: 0.2742 - val_loss: 1.8439 - val_accuracy: 0.3567
Epoch 7/60
225/225 [==============================] - 10s

## ETAPA 4: Avaliação e Predição (Pós-Treinamento)

As células abaixo são as células de avaliação e predição.

Análise manual de um *run* específico.

**Para usar:**
1.  Deixe o loop da ETAPA 3 rodar e gerar as pastas de experimento.
2.  Escolha o experimento que você quer analisar (ex: `meus_experimentos_cnn/run_baseline...`).
3.  Copie o caminho para o arquivo `modelo_melhor.h5` ou `modelo_final.h5` desse experimento.
4.  Cole o caminho nas variáveis `CAMINHO_DO_MODELO_...` nas células abaixo e rode-as.

In [ ]:
### --- AVALIAÇÃO MANUAL DE UM RUN ESPECÍFICO ---

# 1. DEFINA O CAMINHO PARA OS PESOS QUE VOCÊ QUER AVALIAR:
# Ex: "meus_experimentos_cnn/run_modelo_resnet50.../modelo_melhor_weights.h5"
CAMINHO_DOS_PESOS_PARA_AVALIAR = "meus_experimentos_cnn/run_modelo_vgg16_amostras_1000_batch_32_epochs_40/modelo_final_weights.h5"

# 2. DEFINA O TIPO DE MODELO
# 'baseline', 'resnet50', 'efficientnet', ou 'vgg16'
TIPO_DO_MODELO = "vgg16"


# --- O restante do código de avaliação ---

# NOTA: Este bloco funcionará se você NÃO reiniciou o kernel após o treino.
# Ele depende das variáveis 'X_val', 'y_val', e 'label_encoder' da célula do loop.

try:
    y_true = np.argmax(y_val, axis=1)

    print(f"Avaliando o modelo: {CAMINHO_DOS_PESOS_PARA_AVALIAR}")
    print("Limpando sessão da GPU...")
    tf.keras.backend.clear_session()
    
    # --- 1. Recriar a arquitetura do modelo ---
    model_eval = None
    X_eval_data = None
    
    if TIPO_DO_MODELO == 'baseline':
        input_shape = X_val.shape[1:]
        model_eval = criar_modelo_baseline(input_shape, NUM_CLASSES)
        X_eval_data = X_val
    else:
        # Modelos pré-treinados (resnet, efficientnet, vgg16)
        # Recria o X_val_rgb (precisa do X_val da célula anterior)
        X_eval_data = np.repeat(X_val, 3, -1) 
        input_shape_rgb = X_eval_data.shape[1:]
        
        if TIPO_DO_MODELO == 'resnet50':
            model_eval = criar_modelo_resnet50(input_shape_rgb, NUM_CLASSES)
        elif TIPO_DO_MODELO == 'efficientnet':
            model_eval = criar_modelo_efficientnet(input_shape_rgb, NUM_CLASSES)
        elif TIPO_DO_MODELO == 'vgg16':
            model_eval = criar_modelo_vgg16(input_shape_rgb, NUM_CLASSES)

    if model_eval is None:
        print(f"ERRO: Tipo de modelo '{TIPO_DO_MODELO}' não reconhecido.")
    else:
        # --- 2. Carregar os pesos salvos ---
        model_eval.load_weights(CAMINHO_DOS_PESOS_PARA_AVALIAR)
        print("Arquitetura recriada e pesos carregados.")

        # --- 3. Fazer predições ---
        y_pred = model_eval.predict(X_eval_data)
        y_pred_classes = np.argmax(y_pred, axis=1)

        # Relatório de Classificação
        print("\n--- RELATÓRIO DE CLASSIFICAÇÃO ---")
        print(classification_report(y_true, y_pred_classes, target_names=class_names))

        # Matriz de Confusão
        cm = confusion_matrix(y_true, y_pred_classes)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
        plt.title(f'Matriz de Confusão - {TIPO_DO_MODELO}')
        plt.ylabel('Verdadeiro')
        plt.xlabel('Predito')
        plt.show()

except NameError as e:
    print(f"ERRO: Variáveis de dados (X_val, y_val, label_encoder) não encontradas. {e}")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

ERRO: Variáveis de dados (X_val, y_val, label_encoder) não encontradas. name 'y_val' is not defined
